# 04 · Single-Shot Authorized Held-Out Test Evaluation (All 3 Modalities · 6 Tasks)

**Pre-Registered One-Shot Final Test Runner** governed by `docs/test_authorization.json` (`commit_sha = 05ae8d24940397c55b2c250a50860560a89888d2`).

- **AKUDENTAL** (2D Panoramic): `66` held-out test films (`akudental_cls`, `akudental_seg`)
- **DenPAR** (2D Periapical): `200` official `Testing` films (`denpar_cls`, `denpar_seg`)
- **DentVoxel** (3D CBCT, canonical RAS+): `20` held-out test scans (`dentvoxel_cls`, `dentvoxel_seg`)

**Protocol Rules:**
- Uses the **exact** locked prompts, **36 / 36** hash-unique locked WSP replies, and validation-frozen budgets/seeds/hyperparameters from `docs/test_authorization.json`.
- **Zero** hyperparameter changes, **zero** reply selection, **zero** retries.
- Checkpoints incrementally to Google Drive (`My Drive/dental-wsp-runs/sealed_test_v1/`) so transient Colab disconnects can finish the same single pass safely.
- Step 5 bundles all final test CSVs + figures into `dental_wsp_sealed_test_bundle.zip` and writes `final_test_exposure.private.json` to permanently lock against re-execution.
- Clear all notebook outputs before committing to Git.

## Step 1 · Setup, Authorization Verification, Exposure Guard & Lock Integrity Check

In [ ]:
from pathlib import Path
import datetime as dt
import json
import sys
import zipfile

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/WSP_Dental_Data")
    candidates = [Path("/content/dental-wsp"), Path("/content/drive/MyDrive/dental-wsp")]
    REPO_DIR = next((p for p in candidates if (p / "benchmarks").is_dir()), candidates[1])
    OUT_ROOT = Path("/content/drive/MyDrive/dental-wsp-runs/sealed_test_v1")
    VAL_DENTVOXEL_SPLIT = Path("/content/drive/MyDrive/dental-wsp-runs/dentvoxel_v2_ras/dentvoxel_splits.private.json")
else:
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "benchmarks").is_dir() else Path.cwd()
    DATA_ROOT = REPO_DIR / "data"
    OUT_ROOT = REPO_DIR / "runs/sealed_test_v1"
    VAL_DENTVOXEL_SPLIT = REPO_DIR / "runs/dentvoxel_v2_ras/dentvoxel_splits.private.json"

AKU_SRC = DATA_ROOT / "AKUDENTAL"
DENPAR_SRC = DATA_ROOT / "DenPAR Radiographs Dataset.zip"
DENTVOXEL_SRC = DATA_ROOT / "DentVoxel.zip"
AUTH_FILE = REPO_DIR / "docs/test_authorization.json"

assert (REPO_DIR / "benchmarks").is_dir(), f"dental-wsp not found at {REPO_DIR}"
assert AUTH_FILE.is_file(), f"Missing test authorization receipt at {AUTH_FILE}"
assert AKU_SRC.is_dir(), f"Missing AKUDENTAL folder at {AKU_SRC}"
assert DENPAR_SRC.exists(), f"Missing DenPAR archive at {DENPAR_SRC}"
assert DENTVOXEL_SRC.exists(), f"Missing DentVoxel archive at {DENTVOXEL_SRC}"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

OUT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
import pandas as pd
import matplotlib.pyplot as plt

from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.guards import exposure_guard, require_split_allowed
from wsp_core.registry import TASKS
from wsp_core.schemas import sha256_file, strict_json_loads, verify_response_lock
from wsp_core.training.runner import run_classification_three_arm_validation, run_segmentation_three_arm_validation
from wsp_core.visualization import plot_four_arm_comparison, plot_frozen_replies_grid, select_deterministic_examples
from benchmarks.dentvoxel.adapter import get_dentvoxel_scan_ids, make_dentvoxel_scan_splits

# Halt immediately if a completed final test exposure marker already exists
exposure_guard(OUT_ROOT)
exposure_guard(DATA_ROOT)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AUTH = strict_json_loads(AUTH_FILE.read_bytes())
SCHED = AUTH["frozen_schedules"]

# Verify all 6 tasks are explicitly authorized and all 36 locked replies are valid & unique
for task, spec in TASKS.items():
    require_split_allowed("test", task=task, allow_test=True, authorization_file=AUTH_FILE)
    d_path = spec.design_lock_path(REPO_DIR)
    r_path = d_path.with_name(f"{task}.responses.lock.json")
    d_bytes = d_path.read_bytes()
    r_obj = strict_json_loads(r_path.read_bytes())
    valid = verify_response_lock(strict_json_loads(d_bytes), d_bytes, r_obj, spec.architecture)
    assert len(valid) == 6, f"Expected 6 valid locked replies for {task}"
    assert len({rec["raw_sha256"] for rec in r_obj["attempts"].values()}) == 6, f"Duplicate reply in {task}"

# Reuse the exact locked 60/20/20 DentVoxel split from validation (or regenerate identically with seed=2026)
DENTVOXEL_SPLIT = OUT_ROOT / "dentvoxel_splits.private.json"
if VAL_DENTVOXEL_SPLIT.is_file():
    DENTVOXEL_SPLIT.write_bytes(VAL_DENTVOXEL_SPLIT.read_bytes())
elif not DENTVOXEL_SPLIT.is_file():
    scan_ids = get_dentvoxel_scan_ids(DENTVOXEL_SRC)
    assert len(scan_ids) == 100
    DENTVOXEL_SPLIT.write_text(json.dumps(make_dentvoxel_scan_splits(scan_ids, seed=2026), indent=2), encoding="utf-8")
split_plan = json.loads(DENTVOXEL_SPLIT.read_text(encoding="utf-8"))
split_counts = {sp: sum(1 for r in split_plan["rows"] if r["split"] == sp) for sp in ("train", "val", "test")}
assert split_counts == {"train": 60, "val": 20, "test": 20}, f"Unexpected DentVoxel splits: {split_counts}"

print(f"Repo:         {REPO_DIR}")
print(f"Device:       {DEVICE}")
if DEVICE != "cuda":
    print("WARNING: Running on CPU! Switch Colab Runtime -> Change runtime type -> T4/L4 GPU or 2D/3D training will take hours.")
print(f"Auth commit:  {AUTH['commit_sha']}")
print(f"Auth sha256:  {sha256_file(AUTH_FILE)[:16]}")
print(f"Output root:  {OUT_ROOT}")
print("All 6 tasks authorized and 36/36 unique valid locks verified.")

## Step 2 · AKUDENTAL Held-Out Test (`66` Panoramics · `akudental_cls` & `akudental_seg`)

In [ ]:
from benchmarks.akudental.adapter import load_akudental_cls_public, load_akudental_seg_public
from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.training.runner import run_classification_three_arm_validation, run_segmentation_three_arm_validation
from wsp_core.visualization import plot_four_arm_comparison, plot_frozen_replies_grid, select_deterministic_examples

AKU_OUT = OUT_ROOT / "akudental"
AKU_OUT.mkdir(parents=True, exist_ok=True)

# --- 2A. akudental_cls ---
aku_train_cls = load_akudental_cls_public(AKU_SRC, split="train")
aku_test_cls = load_akudental_cls_public(AKU_SRC, split="test", allow_test=True, authorization_file=AUTH_FILE)
assert len(aku_train_cls) == 200 and len(aku_test_cls) == 66
print(f"[akudental_cls] train={len(aku_train_cls)}, test={len(aku_test_cls)} (pos={sum(s.label for s in aku_test_cls)})")

aku_cls_d = REPO_DIR / "benchmarks/akudental/locks/akudental_cls.design.lock.json"
aku_cls_r = REPO_DIR / "benchmarks/akudental/locks/akudental_cls.responses.lock.json"
aku_cls_frozen = verify_response_lock(strict_json_loads(aku_cls_d.read_bytes()), aku_cls_d.read_bytes(), strict_json_loads(aku_cls_r.read_bytes()), "cls-lenet2d-v1")
pd.DataFrame([{"Reply": slot, **evaluate_classifier(p, aku_test_cls)} for slot, p in aku_cls_frozen.items()]).set_index("Reply").to_csv(AKU_OUT / "akudental_cls_test_stage1_frozen.csv")

s_cfg = SCHED["akudental_cls"]
_, aku_cls_sum = run_classification_three_arm_validation(
    task="akudental_cls",
    train_samples=aku_train_cls,
    val_samples=aku_test_cls,
    design_lock_path=aku_cls_d,
    responses_lock_path=aku_cls_r,
    budgets=s_cfg["budgets"],
    seeds=s_cfg["seeds"],
    epochs=s_cfg["epochs"],
    lr=s_cfg["lr"],
    device=DEVICE,
    out_dir=AKU_OUT / "akudental_cls",
    resume=True,
    split_label="test",
)
display(aku_cls_sum.round(4))

# --- 2B. akudental_seg ---
aku_train_seg = load_akudental_seg_public(AKU_SRC, split="train")
aku_test_seg = load_akudental_seg_public(AKU_SRC, split="test", allow_test=True, authorization_file=AUTH_FILE)
assert len(aku_train_seg) == 200 and len(aku_test_seg) == 66

aku_seg_d = REPO_DIR / "benchmarks/akudental/locks/akudental_seg.design.lock.json"
aku_seg_r = REPO_DIR / "benchmarks/akudental/locks/akudental_seg.responses.lock.json"
aku_seg_frozen = verify_response_lock(strict_json_loads(aku_seg_d.read_bytes()), aku_seg_d.read_bytes(), strict_json_loads(aku_seg_r.read_bytes()), "pano-wsp82-v1")
rows = []
for slot, payload in aku_seg_frozen.items():
    sw = evaluate_segmentation_swaps(payload, aku_test_seg)
    au = evaluate_segmentation_paired_audit(payload, aku_test_seg)
    rows.append({
        "Reply": slot,
        "Intact Dice (%)": sw["full_cohort_dice_a_pct"],
        "Image-only Dice (%)": sw["full_cohort_dice_b_pct"],
        "Position-only Dice (%)": sw["full_cohort_dice_c_pct"],
        "Swap loss A (pp)": sw["intact"]["swap_loss_pp"],
        "Swap groups": f"{sw['groups']}/{sw['total_groups']}",
        "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
        "Delta BF1 (pp)": au["bf1_gain_pp"],
        "Delta edge recall (pp)": au["edge_recall_gain_pp"],
        "Box IoU A (%)": au["box_iou_a_pct"],
        "Excluded tooth pixels A (%)": au["excluded_tooth_a_pct"],
    })
aku_seg_s1 = pd.DataFrame(rows).set_index("Reply")
aku_seg_s1.to_csv(AKU_OUT / "akudental_seg_test_stage1_frozen.csv")
display(aku_seg_s1.round(4))

s_cfg = SCHED["akudental_seg"]
_, aku_seg_sum = run_segmentation_three_arm_validation(
    task="akudental_seg",
    architecture="pano-wsp82-v1",
    train_samples=aku_train_seg,
    val_samples=aku_test_seg,
    design_lock_path=aku_seg_d,
    responses_lock_path=aku_seg_r,
    budgets=s_cfg["budgets"],
    seeds=s_cfg["seeds"],
    epochs=s_cfg["epochs"],
    lr=s_cfg["lr"],
    batch_size=s_cfg["batch_size"],
    cosine_lr=s_cfg["cosine_lr"],
    device=DEVICE,
    out_dir=AKU_OUT / "akudental_seg",
    resume=True,
    split_label="test",
)
display(aku_seg_sum.round(4))

rep_p = json.loads((AKU_OUT / "akudental_seg/akudental_seg_test_representative_payloads.json").read_text(encoding="utf-8"))
attr = "Dataset: AKUDENTAL (Oz et al., BMC Oral Health 2025, CC BY-NC-SA 4.0) · Deterministic held-out test sample"
for idx, ex in enumerate(select_deterministic_examples(aku_test_seg, n=2), start=1):
    plot_frozen_replies_grid(ex, aku_seg_frozen, title=f"AKUDENTAL Held-Out Test Example {idx} · 6 Frozen Replies", attribution=attr, out_path=AKU_OUT / f"visuals/test_ex{idx}_frozen.png")
    plot_four_arm_comparison(ex, rep_p, title=f"AKUDENTAL Held-Out Test Example {idx} · 4-Arm Comparison (K=200)", attribution=attr, out_path=AKU_OUT / f"visuals/test_ex{idx}_4arms.png")
print("Completed AKUDENTAL held-out test evaluation.")

## Step 3 · DenPAR Held-Out Test (`200` Official Testing Periapicals · `denpar_cls` & `denpar_seg`)

In [ ]:
from benchmarks.denpar.adapter import load_denpar_cls, load_denpar_seg

DENPAR_OUT = OUT_ROOT / "denpar"
DENPAR_OUT.mkdir(parents=True, exist_ok=True)

# --- 3A. denpar_cls ---
dp_train_cls = load_denpar_cls(DENPAR_SRC, split="train")
dp_test_cls = load_denpar_cls(DENPAR_SRC, split="test", allow_test=True, authorization_file=AUTH_FILE)
assert len(dp_train_cls) == 650 and len(dp_test_cls) == 200
print(f"[denpar_cls] train={len(dp_train_cls)}, test={len(dp_test_cls)} (Upper={sum(s.label for s in dp_test_cls)})")

dp_cls_d = REPO_DIR / "benchmarks/denpar/locks/denpar_cls.design.lock.json"
dp_cls_r = REPO_DIR / "benchmarks/denpar/locks/denpar_cls.responses.lock.json"
dp_cls_frozen = verify_response_lock(strict_json_loads(dp_cls_d.read_bytes()), dp_cls_d.read_bytes(), strict_json_loads(dp_cls_r.read_bytes()), "cls-lenet2d-v1")
pd.DataFrame([{"Reply": slot, **evaluate_classifier(p, dp_test_cls)} for slot, p in dp_cls_frozen.items()]).set_index("Reply").to_csv(DENPAR_OUT / "denpar_cls_test_stage1_frozen.csv")

s_cfg = SCHED["denpar_cls"]
_, dp_cls_sum = run_classification_three_arm_validation(
    task="denpar_cls",
    train_samples=dp_train_cls,
    val_samples=dp_test_cls,
    design_lock_path=dp_cls_d,
    responses_lock_path=dp_cls_r,
    budgets=s_cfg["budgets"],
    seeds=s_cfg["seeds"],
    epochs=s_cfg["epochs"],
    lr=s_cfg["lr"],
    device=DEVICE,
    out_dir=DENPAR_OUT / "denpar_cls",
    resume=True,
    split_label="test",
)
display(dp_cls_sum.round(4))

# --- 3B. denpar_seg ---
dp_train_seg = load_denpar_seg(DENPAR_SRC, split="train")
dp_test_seg = load_denpar_seg(DENPAR_SRC, split="test", allow_test=True, authorization_file=AUTH_FILE)
assert len(dp_train_seg) == 650 and len(dp_test_seg) == 200

dp_seg_d = REPO_DIR / "benchmarks/denpar/locks/denpar_seg.design.lock.json"
dp_seg_r = REPO_DIR / "benchmarks/denpar/locks/denpar_seg.responses.lock.json"
dp_seg_frozen = verify_response_lock(strict_json_loads(dp_seg_d.read_bytes()), dp_seg_d.read_bytes(), strict_json_loads(dp_seg_r.read_bytes()), "periapical-wsp82-v1")
rows = []
for slot, payload in dp_seg_frozen.items():
    sw = evaluate_segmentation_swaps(payload, dp_test_seg)
    au = evaluate_segmentation_paired_audit(payload, dp_test_seg)
    rows.append({
        "Reply": slot,
        "Intact Dice (%)": sw["full_cohort_dice_a_pct"],
        "Image-only Dice (%)": sw["full_cohort_dice_b_pct"],
        "Position-only Dice (%)": sw["full_cohort_dice_c_pct"],
        "Swap loss A (pp)": sw["intact"]["swap_loss_pp"],
        "Swap groups": f"{sw['groups']}/{sw['total_groups']}",
        "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
        "Delta BF1 (pp)": au["bf1_gain_pp"],
        "Delta edge recall (pp)": au["edge_recall_gain_pp"],
        "Box IoU A (%)": au["box_iou_a_pct"],
        "Excluded tooth pixels A (%)": au["excluded_tooth_a_pct"],
    })
dp_seg_s1 = pd.DataFrame(rows).set_index("Reply")
dp_seg_s1.to_csv(DENPAR_OUT / "denpar_seg_test_stage1_frozen.csv")
display(dp_seg_s1.round(4))

s_cfg = SCHED["denpar_seg"]
_, dp_seg_sum = run_segmentation_three_arm_validation(
    task="denpar_seg",
    architecture="periapical-wsp82-v1",
    train_samples=dp_train_seg,
    val_samples=dp_test_seg,
    design_lock_path=dp_seg_d,
    responses_lock_path=dp_seg_r,
    budgets=s_cfg["budgets"],
    seeds=s_cfg["seeds"],
    epochs=s_cfg["epochs"],
    lr=s_cfg["lr"],
    batch_size=s_cfg["batch_size"],
    cosine_lr=s_cfg["cosine_lr"],
    device=DEVICE,
    out_dir=DENPAR_OUT / "denpar_seg",
    resume=True,
    split_label="test",
)
display(dp_seg_sum.round(4))

rep_p = json.loads((DENPAR_OUT / "denpar_seg/denpar_seg_test_representative_payloads.json").read_text(encoding="utf-8"))
attr = "Dataset: DenPAR (Rasnayaka et al., Scientific Data 2025, Zenodo) · Deterministic held-out test sample"
for idx, ex in enumerate(select_deterministic_examples(dp_test_seg, n=2), start=1):
    plot_frozen_replies_grid(ex, dp_seg_frozen, title=f"DenPAR Held-Out Test Example {idx} · 6 Frozen Replies", attribution=attr, out_path=DENPAR_OUT / f"visuals/test_ex{idx}_frozen.png")
    plot_four_arm_comparison(ex, rep_p, title=f"DenPAR Held-Out Test Example {idx} · 4-Arm Comparison (K=200)", attribution=attr, out_path=DENPAR_OUT / f"visuals/test_ex{idx}_4arms.png")
print("Completed DenPAR held-out test evaluation.")

## Step 4 · DentVoxel 3D CBCT Held-Out Test (`20` Canonical RAS+ Scans · `dentvoxel_cls` & `dentvoxel_seg`)

In [ ]:
from benchmarks.dentvoxel.adapter import load_dentvoxel_cls, load_dentvoxel_seg

DV_OUT = OUT_ROOT / "dentvoxel"
DV_OUT.mkdir(parents=True, exist_ok=True)

# --- 4A. dentvoxel_cls ---
dv_train_cls = load_dentvoxel_cls(DENTVOXEL_SRC, DENTVOXEL_SPLIT, split="train", norm_mode="hu_window", stride=4)
dv_test_cls = load_dentvoxel_cls(DENTVOXEL_SRC, DENTVOXEL_SPLIT, split="test", norm_mode="hu_window", stride=4, allow_test=True, authorization_file=AUTH_FILE)
assert len({s.group_id for s in dv_train_cls}) == 60 and len({s.group_id for s in dv_test_cls}) == 20
print(f"[dentvoxel_cls] train={len(dv_train_cls)} (60 scans), test={len(dv_test_cls)} (20 scans, pos={sum(s.label for s in dv_test_cls)})")

dv_cls_d = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_cls.design.lock.json"
dv_cls_r = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_cls.responses.lock.json"
dv_cls_frozen = verify_response_lock(strict_json_loads(dv_cls_d.read_bytes()), dv_cls_d.read_bytes(), strict_json_loads(dv_cls_r.read_bytes()), "cls-lenet2d-v1")
pd.DataFrame([{"Reply": slot, **evaluate_classifier(p, dv_test_cls)} for slot, p in dv_cls_frozen.items()]).set_index("Reply").to_csv(DV_OUT / "dentvoxel_cls_test_stage1_frozen.csv")

s_cfg = SCHED["dentvoxel_cls"]
_, dv_cls_sum = run_classification_three_arm_validation(
    task="dentvoxel_cls",
    train_samples=dv_train_cls,
    val_samples=dv_test_cls,
    design_lock_path=dv_cls_d,
    responses_lock_path=dv_cls_r,
    budgets=s_cfg["budgets"],
    seeds=s_cfg["seeds"],
    epochs=s_cfg["epochs"],
    lr=s_cfg["lr"],
    device=DEVICE,
    out_dir=DV_OUT / "dentvoxel_cls",
    resume=True,
    split_label="test",
)
display(dv_cls_sum.round(4))

# --- 4B. dentvoxel_seg ---
dv_train_seg = load_dentvoxel_seg(DENTVOXEL_SRC, DENTVOXEL_SPLIT, split="train", norm_mode="hu_window")
dv_test_seg = load_dentvoxel_seg(DENTVOXEL_SRC, DENTVOXEL_SPLIT, split="test", norm_mode="hu_window", allow_test=True, authorization_file=AUTH_FILE)
assert len(dv_train_seg) == 60 and len(dv_test_seg) == 20

dv_seg_d = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_seg.design.lock.json"
dv_seg_r = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_seg.responses.lock.json"
dv_seg_frozen = verify_response_lock(strict_json_loads(dv_seg_d.read_bytes()), dv_seg_d.read_bytes(), strict_json_loads(dv_seg_r.read_bytes()), "cbct-wsp82-v1")
rows = []
for slot, payload in dv_seg_frozen.items():
    sw = evaluate_segmentation_swaps(payload, dv_test_seg)
    au = evaluate_segmentation_paired_audit(payload, dv_test_seg)
    rows.append({
        "Reply": slot,
        "Intact 3D Dice (%)": sw["full_cohort_dice_a_pct"],
        "Image-only 3D Dice (%)": sw["full_cohort_dice_b_pct"],
        "Position-only 3D Dice (%)": sw["full_cohort_dice_c_pct"],
        "3D Swap loss A (pp)": sw["intact"]["swap_loss_pp"],
        "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
        "Delta 3D Surface F1 (pp)": au["bf1_gain_pp"],
        "Delta 3D Surface Recall (pp)": au["edge_recall_gain_pp"],
        "3D Box IoU A (%)": au["box_iou_a_pct"],
        "Excluded tooth voxels A (%)": au["excluded_tooth_a_pct"],
    })
dv_seg_s1 = pd.DataFrame(rows).set_index("Reply")
dv_seg_s1.to_csv(DV_OUT / "dentvoxel_seg_test_stage1_frozen.csv")
display(dv_seg_s1.round(4))

s_cfg = SCHED["dentvoxel_seg"]
_, dv_seg_sum = run_segmentation_three_arm_validation(
    task="dentvoxel_seg",
    architecture="cbct-wsp82-v1",
    train_samples=dv_train_seg,
    val_samples=dv_test_seg,
    design_lock_path=dv_seg_d,
    responses_lock_path=dv_seg_r,
    budgets=s_cfg["budgets"],
    seeds=s_cfg["seeds"],
    epochs=s_cfg["epochs"],
    lr=s_cfg["lr"],
    batch_size=s_cfg["batch_size"],
    cosine_lr=s_cfg["cosine_lr"],
    device=DEVICE,
    out_dir=DV_OUT / "dentvoxel_seg",
    resume=True,
    split_label="test",
)
display(dv_seg_sum.round(4))

rep_p = json.loads((DV_OUT / "dentvoxel_seg/dentvoxel_seg_test_representative_payloads.json").read_text(encoding="utf-8"))
attr = "Dataset: DentVoxel 3D CBCT (Figshare, 100 NIfTI scans, 38 structures) · Deterministic held-out test scan"
for idx, ex in enumerate(select_deterministic_examples(dv_test_seg, n=2), start=1):
    for axis in ("axial", "coronal"):
        plot_frozen_replies_grid(ex, dv_seg_frozen, title=f"DentVoxel Held-Out Test Scan {idx} ({axis.title()}) · 6 Frozen Replies", attribution=attr, slice_axis=axis, out_path=DV_OUT / f"visuals/test_scan{idx}_{axis}_frozen.png")
        plot_four_arm_comparison(ex, rep_p, title=f"DentVoxel Held-Out Test Scan {idx} ({axis.title()}) · 4-Arm Comparison (K=60)", attribution=attr, slice_axis=axis, out_path=DV_OUT / f"visuals/test_scan{idx}_{axis}_4arms.png")
print("Completed DentVoxel held-out test evaluation.")

## Step 5 · Bundle All Test Results & Permanently Seal Exposure Marker

In [ ]:
expected_files = [
    OUT_ROOT / "akudental/akudental_cls_test_stage1_frozen.csv",
    OUT_ROOT / "akudental/akudental_cls/akudental_cls_test_summary.csv",
    OUT_ROOT / "akudental/akudental_seg_test_stage1_frozen.csv",
    OUT_ROOT / "akudental/akudental_seg/akudental_seg_test_summary.csv",
    OUT_ROOT / "denpar/denpar_cls_test_stage1_frozen.csv",
    OUT_ROOT / "denpar/denpar_cls/denpar_cls_test_summary.csv",
    OUT_ROOT / "denpar/denpar_seg_test_stage1_frozen.csv",
    OUT_ROOT / "denpar/denpar_seg/denpar_seg_test_summary.csv",
    OUT_ROOT / "dentvoxel/dentvoxel_cls_test_stage1_frozen.csv",
    OUT_ROOT / "dentvoxel/dentvoxel_cls/dentvoxel_cls_test_summary.csv",
    OUT_ROOT / "dentvoxel/dentvoxel_seg_test_stage1_frozen.csv",
    OUT_ROOT / "dentvoxel/dentvoxel_seg/dentvoxel_seg_test_summary.csv",
]
missing = [str(p) for p in expected_files if not p.is_file()]
assert not missing, f"Missing completed test CSVs before sealing: {missing}"

# Package all CSVs, JSON payloads, and PNG plots (excluding private split/exposure files) into one ZIP for download
bundle_path = OUT_ROOT / "dental_wsp_sealed_test_bundle.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_ROOT.rglob("*")):
        if not p.is_file() or p == bundle_path:
            continue
        if p.name.endswith(".private.json") or "final_test_exposure" in p.name:
            continue
        zf.write(p, arcname=str(p.relative_to(OUT_ROOT)))

# Permanently write the private test-exposure marker so exposure_guard blocks accidental re-runs
marker = {
    "schema_version": "dental-wsp-test-exposure-v1",
    "completed_at_utc": dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat(),
    "authorization_sha256": sha256_file(AUTH_FILE),
    "commit_sha": AUTH["commit_sha"],
    "tasks": AUTH["authorized_tasks"],
    "artifact_sha256": {str(p.relative_to(OUT_ROOT)): sha256_file(p) for p in expected_files},
}
marker_path = OUT_ROOT / "final_test_exposure.private.json"
marker_path.write_text(json.dumps(marker, indent=2) + "\n", encoding="utf-8")

print(f"Sealed test bundle saved to: {bundle_path} ({bundle_path.stat().st_size / 1e6:.2f} MB)")
print(f"Permanent exposure seal written to: {marker_path}")